# Import Libraries

In [44]:
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
import torch
import torch.nn as nn
from tqdm import tqdm
import os
from torchvision.transforms import ConvertImageDtype
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import cv2

# Constants

In [45]:
"""
DIRECTORIES
"""
DATASET_PATH = "shaunthesheep/microsoft-catsvsdogs-dataset"

CATS_DIRECTORY = "PetImages/Cat"
DOGS_DIRECTORY = "PetImages/Dog"

TRAIN_DIRECTORY = "dataset/train"
VAL_DIRECTORY = "dataset/val"
TEST_DIRECTORY = "dataset/test"

TRAIN_CATS_DIRECTORY = "dataset/train/cats"
VAL_CATS_DIRECTORY = "dataset/val/cats"
TEST_CATS_DIRECTORY = "dataset/test/cats"
TRAIN_DOGS_DIRECTORY = "dataset/train/dogs"
VAL_DOGS_DIRECTORY = "dataset/train/dogs"
TEST_DOGS_DIRECTORY = "dataset/test/dogs"

JPG_EXTENSION_FILTER = ".jpg"

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

# Useful stuff

In [46]:
os.chdir(r'D:\Vova\Programming\Python projects\dogs-and-cats-classification')

In [47]:
import os

def get_images_with_extension(class_dir: str, extension: str = JPG_EXTENSION_FILTER):
    img_list = [
    img for img in os.listdir(class_dir)
    if img.endswith(extension) and os.path.isfile(os.path.join(class_dir, img))
    ]
    
    return img_list



# Dataset class

In [48]:
class CatDogDataset(Dataset):
    def __init__(self, imgs_dir, transform=None, target_transform=None):
        self.imgs_dir = imgs_dir
        self.transform = transform
        self.target_transform = target_transform
        self.images = []
        self.labels = []

        try:
            for label, class_name in enumerate(['cats', 'dogs']):
                class_dir = os.path.join(imgs_dir, class_name)

                for i, img_name in enumerate(get_images_with_extension(class_dir)):
                    self.images.append(os.path.join(class_dir, img_name))
                    self.labels.append(label)
                    if i % 1000 == 0:
                        print(f"Collected {i} images")
                    
        except Exception as e:
            print(f"Folders scanning error: {e}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image_path = self.images[index]
        image = Image.open(image_path).convert('RGB')
        label = self.labels[index]

        if self.transform:
            image = self.transform(image)
        
        if self.target_transform:
            label = self.target_transform(label)

        return image, label


# CNN architecture

In [49]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel=3, stride=1, padding=1):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel, stride, padding),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class CatDogCNN(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()

        # Feature extractor
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 32),
            nn.MaxPool2d(2),

            ConvBlock(32, 64),
            ConvBlock(64, 64),  
            nn.MaxPool2d(2),

            ConvBlock(64, 128),
            nn.MaxPool2d(2)
        )

        # Global pooler
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


# Training Loop

In [50]:
def train_one_epoch(model, loader, optimizer, criterion, device):

    model.train()
    running_loss = 0
    
    for images, labels in loader:

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    
    return running_loss / len(loader)
    

# Validation Loop

In [51]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():

        for images, labels in loader:

            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(images, labels)

            total_loss += loss.items()

            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total = labels.size(0)
    
    avg_loss = total_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy

# Early stopping logic

In [52]:
class EarlyStopping():
    def __init__(self, patience=3, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')

    def __call__(self, validation_loss):
        if validation_loss < self.best_loss:
            self.best_loss = validation_loss
            self.counter = 0
        
        if validation_loss > (self.best_loss + self.min_delta):
            self.counter += 1
            
        
        if self.counter >= self.patience:
            return True
        return False

# Model saving logic

In [53]:
class ModelCheckpoint():
    def __init__(self, model, filepath, mode: str='min', verbose: bool=False):
        """
        model: model that needs saving
        filepath: path for saving (ex. path.pt)
        mode: 'min' - save when loss decreasing
              'max' - save when loss increasing
        verbose: show saving info
        """
        self.model = model
        self.filepath = filepath
        self.mode = mode
        self.verbose = verbose
        self.best_loss = float('inf') if mode == 'min' else -float('inf')

    def __call__(self, current_loss):
        if (self.mode == 'min' and current_loss < self.best_loss) or (self.mode == 'max' and current_loss > self.best_loss):

            self.best_loss = current_loss
            torch.save(self.model.state_dict(), self.filepath)

            if self.verbose:
                print(f"Model was saved. Best loss: {self.best_loss:.2f}")

# Training model

In [ ]:
def train_catdog_classifier(
        model, 
        train_loader, 
        val_loader, 
        criterion, 
        optimizer, 
        device, 
        epochs=20
    ):

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": []
    }

    checkpoint = ModelCheckpoint(model, 'best_model.pt', verbose=True)
    early_stopping = EarlyStopping(patience=5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        patience=3
    )

    for epoch in range(epochs):

        train_loss = train_one_epoch(
            model, 
            train_loader,
            optimizer, 
            criterion, 
            device
        )
        
        val_loss, val_accuracy = validate_one_epoch(
            model, 
            val_loader,  
            criterion, 
            device
        )

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        # Epoch's statistics
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train loss: {train_loss:.4f}")
        print(f"Validation loss: {val_loss:.4f}")
        print(f"Validation Accuracy: {val_accuracy*100:.2f}%")
        
        # Saving model if the current model is better that the previous one
        checkpoint(val_loss)
        
        # Using early stopping to train with optimal epochs
        if early_stopping(val_loss):
            print(f"Early Stopping on {epoch} epoch.")
            break

    return history
        

In [55]:
img_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
])

train_dataset = CatDogDataset(TRAIN_DIRECTORY, transform=img_transform)
val_dataset = CatDogDataset(VAL_DIRECTORY, transform=img_transform)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

# Using GPU for more faster model training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CatDogCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)



total_params = sum(p.numel() for p in model.parameters())
print(f"Model's params: {total_params}")

Collected 0 images
Collected 1000 images
Collected 2000 images
Collected 3000 images
Collected 4000 images
Collected 5000 images
Collected 6000 images
Collected 7000 images
Collected 8000 images
Collected 9000 images
Collected 10000 images
Collected 11000 images
Collected 12000 images
Collected 0 images
Collected 1000 images
Collected 2000 images
Collected 3000 images
Collected 4000 images
Collected 5000 images
Collected 6000 images
Collected 7000 images
Collected 8000 images
Collected 9000 images
Collected 10000 images
Collected 11000 images
Collected 12000 images
Collected 0 images
Collected 1000 images
Collected 2000 images
Collected 3000 images
Collected 0 images
Collected 1000 images
Model's params: 173602


In [ ]:
losses, accuracies = train_catdog_classifier(model, train_loader, val_loader, criterion, optimizer, device, epochs=3)